# Local Diabatization


## Table of Content: <a name="TOC"></a>
1. [Theory](#1)

   1.1. [Local Diabatization - the idea](#1.1)
   
   1.2. [Polar decomposition approach](#1.2)
   
   1.3. [Optimality](#1.3)
   
   1.4. [Final expression](#1.4)
   
   1.5. [Handling ill-conditioned (dropped) states](#1.5)
   
   1.6. [Reduced-subspace construction](#1.6)
      
   1.7. [Partial isometry](#1.7)
   
   1.8. [Physical interpretation](#1.8)


2. [Now testing](#2)

   2.1. [Trivial case](#2.1)
   
   2.2. [Already unitary matrix](#2.2)
   
   2.3. [Random non-unitary matrix](#2.3)
   
   2.4. [Nearly singular overlap inverse (numerical stability)](#2.4)
   
   2.5. [Nearly singular with drop](#2.5)
   
   2.6. [Phase consistency (diagonal matrix)](#2.6)
   
   2.7. [Gauge optimality (closest unitary)](#2.7)


3. [Some examples of computing projection matrices](#3)

   3.1. [Simple, nearly-unity matrix](#3.1)
   
   3.2. [Potental state-reordering in 2-level system](#3.2)
   
   3.3. [Near degeneracy](#3.3)


## A. Learning objectives

- to understand theory and practice of local diabatization-related functions


## B. Use cases

- [N/A]


## C. Functions

- `libra_py`
  - `dyn`
    - `local_diabatization`
      - [`orthogonalize_T`](#orthogonalize_T-1)

In [1]:
from liblibra_core import *
import libra_py.dyn.local_diabatization as ld
import pytest
import numpy as np

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

## 1. Theory 
[Back to TOC](#TOC)
<a name="1"></a>

### 1.1. Local Diabatization - the idea
[Back to TOC](#TOC)
<a name="1.1"></a>

We are given a generally **non-unitary** matrix

$$
X = P^{-1}(t, t+\Delta t),
$$

where the overlap matrix between adiabatic states at consecutive time steps is

$$
P(t, t+\Delta t)
= \langle \psi_{\mathrm{adi}}(t) \mid \psi_{\mathrm{adi}}(t+\Delta t) \rangle.
$$

We define a transformed adiabatic basis as

$$
\lvert \tilde{\psi}_{\mathrm{adi}}(t+\Delta t) \rangle
=
\lvert \psi_{\mathrm{adi}}(t+\Delta t) \rangle \, T(t+\Delta t),
$$

and require that its overlap with the previous basis be exactly the identity:

$$
\langle \tilde{\psi}_{\mathrm{adi}}(t) \mid
\tilde{\psi}_{\mathrm{adi}}(t+\Delta t) \rangle
= I.
$$

This is what makes the approximation the **diabatization**

Formally, this suggests choosing

$$
T(t+\Delta t) \approx P^{-1}(t, t+\Delta t),
$$

and 

$$
T(t) \approx I.
$$

The last equation is what makes this diabatization approach **local** - we forget about the whole history of previous basis states swaps and phase changes.

### 1.2. Polar decomposition approach

[Back to TOC](#TOC)
<a name="1.2"></a>

But the inverse overlap matrix is **not guaranteed to be unitary** due to numerical noise and finite time steps.
Therefore, we seek a **unitary matrix unitarily equivalent to** $P^{-1}$.

Let

$$
X = P^{-1},
$$

and define the Hermitian, positive-definite matrix

$$
S = X^\dagger X.
$$

We construct the matrix

$$
U = X \, S^{-1/2}.
$$

Then,

$$
\begin{aligned}
U^\dagger U
&= (S^{-1/2})^\dagger X^\dagger X S^{-1/2} \\
&= S^{-1/2} \, S \, S^{-1/2} \\
&= I.
\end{aligned}
$$

Thus, $U$ is **unitary**.

This corresponds to the **right polar decomposition** of $X$:

$$
X = U H,
$$

with

$$
H = (X^\dagger X)^{1/2}.
$$


### 1.3. Optimality

[Back to TOC](#TOC)
<a name="1.3"></a>

Among all unitary matrices, $U$ is the **closest** to $X$ in the Frobenius norm sense:

$$
U = \arg\min_{Q^\dagger Q = I} \| X - Q \|_F.
$$


### 1.4. Final expression

[Back to TOC](#TOC)
<a name="1.4"></a>

The orthonormalized transformation matrix returned by the function is

$$
T_{\mathrm{orth}}
=
P^{-1}
\left(
P^{-1\dagger} P^{-1}
\right)^{-1/2}.
$$

This matrix is unitary and preserves the physical adiabatic overlap structure
while eliminating accumulated non-unitarity during time propagation.


### 1.5. Handling ill-conditioned (dropped) states

[Back to TOC](#TOC)
<a name="1.5"></a>

In practical nonadiabatic dynamics, the overlap matrix

$$
P(t, t+\Delta t)
=
\langle \psi_{\mathrm{adi}}(t) \mid \psi_{\mathrm{adi}}(t+\Delta t) \rangle
$$

may become ill-conditioned due to large time steps, near-degeneracies,
or numerical noise. As a result, its inverse

$$
T = P^{-1}(t, t+\Delta t)
$$

may be nearly singular or rank-deficient. The polar decomposition-based orthonormalization

$$
U = T (T^\dagger T)^{-1/2}
$$

requires the Hermitian matrix

$$
S = T^\dagger T
$$

to be invertible. If one or more eigenvalues of $S$ approach zero, the
inverse square root $S^{-1/2}$ does not exist, and no unitary matrix $U$
satisfying

$$
U^\dagger U = I
$$

can be constructed. This signals a loss of linear independence in the adiabatic basis.


### 1.6. Reduced-subspace construction

[Back to TOC](#TOC)
<a name="1.6"></a>

Let the spectral decomposition of $S$ be

$$
S = \sum_{i=1}^N \lambda_i \, |v_i\rangle \langle v_i|,
$$

with eigenvalues ordered as

$$
\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_N \ge 0.
$$

Define the set of well-conditioned states as those satisfying

$$
\lambda_i > \text{tol}.
$$

Let $r$ be the number of retained states and define the projector onto
the corresponding subspace:

$$
\Pi
=
\sum_{i=1}^r |v_i\rangle \langle v_i|.
$$


### 1.7. Partial isometry

[Back to TOC](#TOC)
<a name="1.7"></a>

Restricting the inverse square root to the retained subspace gives

$$
S^{-1/2}
=
\sum_{i=1}^r \lambda_i^{-1/2} |v_i\rangle \langle v_i|.
$$

The resulting transformation matrix is

$$
U
=
T \, S^{-1/2}.
$$

This matrix is no longer unitary on the full space. Instead, it satisfies

$$
U^\dagger U = \Pi,
$$

meaning that $U$ is a **partial isometry** mapping the original space
onto the well-conditioned subspace.


### 1.8. Physical interpretation

[Back to TOC](#TOC)
<a name="1.8"></a>

Dropping ill-conditioned states corresponds to an explicit reduction of
the electronic Hilbert space. Components of the wavefunction associated
with discarded eigenvectors are removed, and the dynamics proceeds in
the reduced subspace defined by $\Pi$.

This procedure should be used with care, as it changes the dimensionality
and physical content of the electronic problem. In most production
nonadiabatic molecular dynamics simulations, ill-conditioning indicates
the need for a smaller nuclear time step rather than state reduction.

<a name="orthogonalize_T-1"></a>

In [2]:
help(ld.orthogonalized_T)

Help on function orthogonalized_T in module libra_py.dyn.local_diabatization:

orthogonalized_T(T, tol=1e-10, drop=False)
    Orthonormalize a matrix T via (right) polar decomposition.
    
    Parameters
    ----------
    T : (N, N) complex ndarray
        Typically T = P^{-1}(t, t+dt), where P is an adiabatic overlap matrix.
    tol : float
        Eigenvalue cutoff for detecting ill-conditioning.
    drop : bool, optional
        If False (default), require T to be full rank and return a unitary matrix.
        If True, drop ill-conditioned states and return a partial isometry.
    
    Returns
    -------
    U : (N, N) complex ndarray
        Orthonormalized transformation matrix.
        - If drop=False: U is unitary (U† U = I)
        - If drop=True:  U† U is a projector
    
    info : dict (only if drop=True)
        Dictionary with diagnostic information:
        - 'rank'       : number of retained states
        - 'projector'  : U† U
        - 'eigvals'    : eigenvalues of 

A helper function

In [3]:
def is_unitary(U, tol=1e-10):
    I = np.eye(U.shape[0], dtype=U.dtype)
    return np.linalg.norm(U.conj().T @ U - I) < tol

## 2. Now testing 

[Back to TOC](#TOC)
<a name="2"></a>

### 2.1. Trivial case

[Back to TOC](#TOC)
<a name="2.1"></a>

If $T = I$, the result must be exactly $I$

In [4]:
def test_identity_matrix():
    n = 5
    T = np.eye(n, dtype=np.complex128)

    U = ld.orthogonalized_T(T)

    print( np.allclose(U, np.eye(n)) )
    print(is_unitary(U) )

In [5]:
test_identity_matrix()

True
True


### 2.2. Already unitary matrix

[Back to TOC](#TOC)
<a name="2.2"></a>

If $T$ is unitary, orthogonalization should not modify it (up to numerical noise).

In [6]:
def test_unitary_input():
    rng = np.random.default_rng(0)
    n = 4

    # Random unitary via QR
    A = rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))
    Q, _ = np.linalg.qr(A)

    U = ld.orthogonalized_T(Q)

    print(is_unitary(U))
    print(np.allclose(U, Q, atol=1e-10))

In [7]:
test_unitary_input()

True
True


### 2.3. Random non-unitary matrix

[Back to TOC](#TOC)
<a name="2.3"></a>

This is the main expected use case

In [8]:
def test_random_nonunitary_matrix():
    rng = np.random.default_rng(1)
    n = 6

    T = rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))
    U = ld.orthogonalized_T(T)

    # Must be unitary
    print( is_unitary(U) )

    # U should span the same column space as T
    # i.e., T = U H for some Hermitian H
    H = U.conj().T @ T
    print( np.allclose(H, H.conj().T, atol=1e-10))

In [9]:
test_random_nonunitary_matrix()

True
True


### 2.4. Nearly singular overlap inverse (numerical stability)

[Back to TOC](#TOC)
<a name="2.4"></a>

This mimics realistic NA-MD overlaps with near-linear dependence.

In [10]:
def test_near_singular_raises():
    rng = np.random.default_rng(1)
    n = 5

    U0, _ = np.linalg.qr(rng.normal(size=(n, n)))
    V0, _ = np.linalg.qr(rng.normal(size=(n, n)))
    s = np.ones(n)
    s[2] = 1e-8

    T = U0 @ np.diag(s) @ V0.conj().T

    with pytest.raises(ValueError):
        ld.orthogonalized_T(T, tol=1e-6)

In [11]:
test_near_singular_raises()

### 2.5. Nearly singular with drop

[Back to TOC](#TOC)
<a name="2.5"></a>

In [12]:
def test_near_singular_drop():
    rng = np.random.default_rng(2)
    n = 5

    U0, _ = np.linalg.qr(rng.normal(size=(n, n)))
    V0, _ = np.linalg.qr(rng.normal(size=(n, n)))
    s = np.ones(n)
    s[-1] = 1e-8

    T = U0 @ np.diag(s) @ V0.conj().T

    U, info = ld.orthogonalized_T(T, tol=1e-6, drop=True)

    P = info["projector"]
    print(info["rank"])
    print( np.allclose(P @ P, P, atol=1e-10))


In [13]:
test_near_singular_drop()

4
True


### 2.6. Phase consistency (diagonal matrix)

[Back to TOC](#TOC)
<a name="2.6"></a>

For diagonal matrices, the result should be pure phases.

In [14]:
def test_diagonal_matrix():
    phases = np.exp(1j * np.array([0.1, 1.3, -2.0]))
    scales = np.array([2.0, 0.5, 3.0])

    T = np.diag(scales * phases)

    U = ld.orthogonalized_T(T)

    # Result must be diagonal unitary with same phases
    print( is_unitary(U))
    print( np.allclose(np.diag(U), phases) )


In [15]:
test_diagonal_matrix()

True
True


### 2.7. Gauge optimality (closest unitary)

[Back to TOC](#TOC)
<a name="2.7"></a>

Checks Frobenius optimality property

In [16]:
def test_closest_unitary_property():
    rng = np.random.default_rng(3)
    n = 4

    T = rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))
    U = ld.orthogonalized_T(T)

    # Compare against a random unitary
    A = rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))
    Q, _ = np.linalg.qr(A)

    dist_U = np.linalg.norm(T - U)
    dist_Q = np.linalg.norm(T - Q)
    
    print(dist_U)
    print(dist_Q)

    #assert dist_U <= dist_Q + 1e-10

In [17]:
test_closest_unitary_property()

4.948693906336126
6.854242785287475


## 3. Some examples of computing projection matrices

[Back to TOC](#TOC)
<a name="3"></a>

### 3.1. Simple, nearly-unity matrix

[Back to TOC](#TOC)
<a name="3.1"></a>

In [18]:
S = np.array([ [1.00, 0.02],
               [0.03, -0.99]
             ] )

In [19]:
iS = np.linalg.inv(S)
print(iS)

[[ 0.99939431  0.02018978]
 [ 0.03028468 -1.0094892 ]]


In [20]:
proj = ld.orthogonalized_T(iS)

print(proj)

[[ 0.9996845  0.0251177]
 [ 0.0251177 -0.9996845]]


In [21]:
proj.T @ proj

array([[1.00000000e+00, 4.31825104e-17],
       [4.31825104e-17, 1.00000000e+00]])

### 3.2. Potental state-reordering in 2-level system

[Back to TOC](#TOC)
<a name="3.2"></a>

In [22]:
S = np.array([ [0.02, 1.00],
               [-0.99, -0.03]
             ] )
iS = np.linalg.inv(S)
proj = ld.orthogonalized_T(iS)
print(proj)

[[-0.00502506 -0.99998737]
 [ 0.99998737 -0.00502506]]


### 3.3. Near degeneracy

[Back to TOC](#TOC)
<a name="3.3"></a>

In [23]:
S = np.array([ [0.5, 0.5],
               [0.49, 0.48]
             ] )
iS = np.linalg.inv(S)
proj = ld.orthogonalized_T(iS)
print(proj)

[[ 0.0201979  0.999796 ]
 [ 0.999796  -0.0201979]]
